# Applied Robotics and AI Project – Part 2
## CNN object classification on CIFAR-10 (built on Session 8 `CNN_v3`)

**Harish A · BEng (Hons) Robotics and AI · University of Hertfordshire / PSB Academy**

This notebook covers every CNN output the brief asks for:

| # | Brief requirement | Section |
|---|---|---|
| 1 | Show all the layers in the model | 2, 4 |
| 2 | Show the accuracy score | 3, 5, 6 |
| 3 | Prediction on three images of my own | 8 |
| 4 | Prediction on an image taken from the e-puck camera in Webots | 9 |
| – | Optimisation so robot motion is not affected | 10 |

**Run it in Google Colab:** *Runtime → Change runtime type → GPU*, then *Runtime → Run all*.
Section 8 and 9 ask you to upload images (a file picker appears).

## 1. Setup and data
CIFAR-10: 60,000 colour images of 32×32 px in 10 classes, 50,000 for training and 10,000 for testing.
Pixel values are scaled from 0–255 to 0–1, exactly as in Session 8.

In [ ]:
import json, os, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
tf.keras.utils.set_random_seed(42)
os.makedirs("outputs", exist_ok=True)

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images, test_images = train_images / 255.0, test_images / 255.0
train_images = train_images.astype("float32"); test_images = test_images.astype("float32")

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
print("train:", train_images.shape, train_labels.shape, "| test:", test_images.shape)

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.suptitle("CIFAR-10 training samples")
plt.savefig("outputs/cifar10_samples.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Baseline model – the Session 8 `CNN_v3` architecture (all layers)

| Layer | Output shape | What happens |
|---|---|---|
| Conv2D 32 × (3×3), ReLU | 30 × 30 × 32 | kernel slides 30 positions on a 32 px image; one map per filter |
| MaxPooling2D 2×2 | 15 × 15 × 32 | halves width and height, channels unchanged |
| Conv2D 64 × (3×3), ReLU | 13 × 13 × 64 | each filter uses all 32 input channels |
| MaxPooling2D 2×2 | 6 × 6 × 64 | |
| Conv2D 64 × (3×3), ReLU | 4 × 4 × 64 | |
| Flatten | 1024 | 4 × 4 × 64 = 1024 values |
| Dense 64, ReLU | 64 | every neuron sees all 1024 inputs |
| Dense 10 (no softmax) | 10 | raw scores (logits), one per class → `from_logits=True` |

In [ ]:
def build_baseline():
    # Session 8 CNN_v3 model, layer for layer
    model = models.Sequential(name="baseline_cnn_v3")
    model.add(layers.Input(shape=(32, 32, 3)))
    model.add(layers.Conv2D(32, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(10))
    return model

baseline = build_baseline()
baseline.summary()
try:
    tf.keras.utils.plot_model(baseline, "outputs/baseline_layers.png", show_shapes=True,
                              show_layer_names=True, dpi=90)
    from IPython.display import Image, display; display(Image("outputs/baseline_layers.png"))
except Exception as err:
    print("plot_model skipped:", err)

## 3. Training the baseline: SGD vs Adam
Session 8 trained one model with SGD and then re-compiled the **same** model with Adam, so the Adam run
started from SGD's weights. Here each optimiser gets a **fresh** model, so the comparison is fair.
Both use 10 epochs, sparse categorical cross-entropy with `from_logits=True`, and the test set for validation.

In [ ]:
LOSS = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
results = {}

def train_baseline(optimizer_name):
    model = build_baseline()
    model.compile(optimizer=optimizer_name, loss=LOSS, metrics=['accuracy'])
    t0 = time.time()
    history = model.fit(train_images, train_labels, epochs=10,
                        validation_data=(test_images, test_labels), verbose=2)
    minutes = (time.time() - t0) / 60
    test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=0)
    results[f"baseline_{optimizer_name}"] = {
        "test_accuracy": round(float(test_acc), 4), "test_loss": round(float(test_loss), 4),
        "params": int(model.count_params()), "epochs": 10, "train_minutes": round(minutes, 1),
        "best_val_accuracy": round(float(max(history.history['val_accuracy'])), 4)}
    print(optimizer_name, "test accuracy:", round(float(test_acc), 4))
    return model, history

baseline_sgd, hist_sgd = train_baseline('sgd')
baseline_adam, hist_adam = train_baseline('adam')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for hist, name, colour in [(hist_sgd, "SGD", "tab:blue"), (hist_adam, "Adam", "tab:orange")]:
    ax[0].plot(hist.history['accuracy'], color=colour, label=f"{name} train")
    ax[0].plot(hist.history['val_accuracy'], color=colour, ls="--", label=f"{name} validation")
    ax[1].plot(hist.history['loss'], color=colour, label=f"{name} train")
    ax[1].plot(hist.history['val_loss'], color=colour, ls="--", label=f"{name} validation")
ax[0].set_title("Accuracy"); ax[1].set_title("Loss")
for a in ax: a.set_xlabel("Epoch"); a.legend()
plt.tight_layout(); plt.savefig("outputs/baseline_sgd_vs_adam.png", dpi=130); plt.show()
print(json.dumps(results, indent=2))

**Reading the curves.** As in the Session 8 notes, Adam learns faster than SGD. Watch for the point where
training accuracy keeps rising but validation accuracy flattens or validation loss rises: that is
overfitting starting (the notes saw it around epoch 6–7, with training 0.87 vs validation 0.70 at epoch 10).

## 4. Improved model – tackling the overfitting

Changes from the baseline, each aimed at the overfitting seen above:

* **Data augmentation** (random flip, shift, small rotation) – the model sees new variations every epoch.
  It is applied in the input pipeline, *not* inside the model, so the deployed model stays small.
* **Two 3×3 convolutions per block + Batch Normalisation** – deeper features, more stable training.
* **Dropout** after each block and before the output – stops neurons relying on each other.
* **Global average pooling** instead of Flatten – far fewer weights in the dense layer.
* **Early stopping + learning-rate reduction** – stops when validation accuracy stops improving.

In [ ]:
def conv_block(x, filters, dropout):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    return layers.Dropout(dropout)(x)

def build_improved():
    inputs = layers.Input(shape=(32, 32, 3), name="image")
    x = conv_block(inputs, 32, 0.2)
    x = conv_block(x, 64, 0.3)
    x = conv_block(x, 128, 0.4)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, name="logits")(x)        # logits, like the baseline
    return models.Model(inputs, outputs, name="improved_cnn")

improved = build_improved()
improved.summary()
try:
    tf.keras.utils.plot_model(improved, "outputs/improved_layers.png", show_shapes=True, dpi=80)
except Exception as err:
    print("plot_model skipped:", err)

In [ ]:
augment = tf.keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomTranslation(0.1, 0.1),
                               layers.RandomRotation(0.05)])
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (tf.data.Dataset.from_tensor_slices((train_images, train_labels))
            .shuffle(10000, seed=42).batch(64)
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

improved.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=LOSS, metrics=['accuracy'])
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                              restore_best_weights=True),
             tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                                                  min_lr=1e-5)]
t0 = time.time()
hist_imp = improved.fit(train_ds, epochs=40, validation_data=(test_images, test_labels),
                        callbacks=callbacks, verbose=2)
minutes = (time.time() - t0) / 60
test_loss, test_acc = improved.evaluate(test_images, test_labels, verbose=0)
results["improved_adam"] = {
    "test_accuracy": round(float(test_acc), 4), "test_loss": round(float(test_loss), 4),
    "params": int(improved.count_params()), "epochs": len(hist_imp.history['loss']),
    "train_minutes": round(minutes, 1),
    "best_val_accuracy": round(float(max(hist_imp.history['val_accuracy'])), 4)}
print(json.dumps(results["improved_adam"], indent=2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(hist_imp.history['accuracy'], label="train"); ax[0].plot(hist_imp.history['val_accuracy'], label="validation")
ax[1].plot(hist_imp.history['loss'], label="train"); ax[1].plot(hist_imp.history['val_loss'], label="validation")
ax[0].set_title("Improved model – accuracy"); ax[1].set_title("Improved model – loss")
for a in ax: a.set_xlabel("Epoch"); a.legend()
plt.tight_layout(); plt.savefig("outputs/improved_curves.png", dpi=130); plt.show()

## 5. Accuracy comparison

In [ ]:
names = {"baseline_sgd": "Baseline · SGD", "baseline_adam": "Baseline · Adam", "improved_adam": "Improved · Adam"}
print(f"{'Model':<18}{'Test acc':>10}{'Best val':>10}{'Params':>10}{'Epochs':>8}{'Min':>6}")
for k, r in results.items():
    print(f"{names[k]:<18}{r['test_accuracy']:>10.2%}{r['best_val_accuracy']:>10.2%}{r['params']:>10,}{r['epochs']:>8}{r['train_minutes']:>6}")

plt.figure(figsize=(7, 4))
bars = plt.bar([names[k] for k in results], [results[k]["test_accuracy"] * 100 for k in results],
               color=["tab:blue", "tab:orange", "tab:green"])
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5, f"{b.get_height():.1f}%", ha="center")
plt.ylabel("Test accuracy (%)"); plt.ylim(0, 100); plt.title("CIFAR-10 test accuracy")
plt.tight_layout(); plt.savefig("outputs/accuracy_comparison.png", dpi=130); plt.show()

## 6. Error analysis (best model)

In [ ]:
best_name = max(results, key=lambda k: results[k]["test_accuracy"])
best_model = {"baseline_sgd": baseline_sgd, "baseline_adam": baseline_adam, "improved_adam": improved}[best_name]
print("Best model:", names[best_name])

logits = best_model.predict(test_images, verbose=0)
y_pred = logits.argmax(axis=1); y_true = test_labels.flatten()
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=10).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10), class_names, rotation=45, ha="right"); ax.set_yticks(range(10), class_names)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix – {names[best_name]}")
fig.colorbar(im); plt.tight_layout(); plt.savefig("outputs/confusion_matrix.png", dpi=130); plt.show()

per_class = cm.diagonal() / cm.sum(axis=1)
for name, acc in sorted(zip(class_names, per_class), key=lambda t: t[1]):
    print(f"{name:<11}{acc:.1%}")
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"\nMost common mistake: {class_names[i]} predicted as {class_names[j]} ({off[i, j]} times)")
results["per_class_accuracy"] = {n: round(float(a), 3) for n, a in zip(class_names, per_class)}
results["most_confused"] = [class_names[i], class_names[j], int(off[i, j])]

## 7. Save the model
(The Session 8 notebook saved `final_model.keras` but loaded `final_model.h5`; here the same file is used.)

In [ ]:
best_model.save("outputs/final_model.keras")
model_for_prediction = tf.keras.models.load_model("outputs/final_model.keras")
print("Saved and reloaded outputs/final_model.keras")

## 8. Predictions on my own images (3 photos)
Upload three photos of CIFAR-10 objects (for example a car, a cat and a ship). Each image is resized to
32×32 and scaled to 0–1, as in the Session 8 `load_image` function. `sample_image.png` from Session 8 can be
uploaded too.

In [ ]:
def load_image(filename):
    img = tf.keras.utils.load_img(filename, target_size=(32, 32))
    img = tf.keras.utils.img_to_array(img).reshape(1, 32, 32, 3).astype("float32") / 255.0
    return img

def predict_files(paths, title, out_png):
    if not paths:
        print("No images uploaded - re-run this cell and choose some files.")
        return []
    fig, ax = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4.6), squeeze=False)
    found = []
    for k, path in enumerate(paths):
        probs = tf.nn.softmax(model_for_prediction.predict(load_image(path), verbose=0)[0]).numpy()
        top3 = probs.argsort()[::-1][:3]
        ax[0, k].imshow(tf.keras.utils.load_img(path)); ax[0, k].axis("off")
        ax[0, k].set_title("\n".join(f"{class_names[c]}: {probs[c]:.1%}" for c in top3), fontsize=10)
        found.append({"image": os.path.basename(path),
                      "top3": [[class_names[c], round(float(probs[c]), 3)] for c in top3]})
        print(f"{os.path.basename(path):<30} -> {class_names[top3[0]]} ({probs[top3[0]]:.1%})")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(out_png, dpi=130); plt.show()
    return found

def upload_images(folder):
    os.makedirs(folder, exist_ok=True)
    try:
        from google.colab import files
        for name, data in files.upload().items():
            with open(os.path.join(folder, name), "wb") as f:
                f.write(data)
    except ImportError:
        print(f"Not in Colab: put the images in ./{folder}/ instead")
    return sorted(os.path.join(folder, n) for n in os.listdir(folder)
                  if n.lower().endswith((".png", ".jpg", ".jpeg")))

own_paths = upload_images("my_images")
results["own_image_predictions"] = predict_files(own_paths, "Predictions on my own images",
                                                 "outputs/own_image_predictions.png")

## 9. Prediction on the e-puck camera image from Webots
In Webots, open `cnn_object_test.wbt`: miniature versions of a dog, cat, horse, deer and an airplane
(all CIFAR-10 classes) stand at random positions in a walled arena. The e-puck drives up to each and saves `dog.png`, `cat.png`, `horse.png`, `deer.png` and `airplane.png` in
`controllers/epuck_controller/captures/`
(copies are in `cnn/webots_images/`). Upload them here.

Expect lower confidence than on photos: rendered, low-resolution simulation frames look different from the
real photos CIFAR-10 was built from (the *domain gap*).

In [ ]:
webots_paths = upload_images("webots_images")
results["webots_predictions"] = predict_files(webots_paths, "Predictions on e-puck camera frames (Webots)",
                                              "outputs/webots_predictions.png")

## 10. Optimisation for the robot – post-training quantisation (TFLite)
The robot's control loop runs every 32 ms, so inference must be small and fast. The model is converted to
TensorFlow Lite in three forms and each is measured on the same 1,000 test images, one image at a time
(as on the robot):

* **fp32** – plain conversion
* **fp16** – weights stored as 16-bit floats (≈2× smaller)
* **int8** – weights and activations as 8-bit integers (≈4× smaller, fastest on CPU)

In [ ]:
def to_tflite(model, mode):
    try:
        conv = tf.lite.TFLiteConverter.from_keras_model(model)
    except Exception:
        model.export("outputs/saved_model")
        conv = tf.lite.TFLiteConverter.from_saved_model("outputs/saved_model")
    if mode in ("fp16", "int8"):
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if mode == "fp16":
        conv.target_spec.supported_types = [tf.float16]
    if mode == "int8":
        conv.representative_dataset = lambda: ([train_images[i:i + 1]] for i in range(300))
    path = f"outputs/cifar10_cnn_{mode}.tflite"
    with open(path, "wb") as f:
        f.write(conv.convert())
    return path

def bench_tflite(path, n=1000):
    interp = tf.lite.Interpreter(model_path=path, num_threads=1); interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    correct, times = 0, []
    for img, label in zip(test_images[:n], test_labels[:n].flatten()):
        interp.set_tensor(inp["index"], img[None].astype(inp["dtype"]))
        t0 = time.perf_counter(); interp.invoke(); times.append(time.perf_counter() - t0)
        correct += int(interp.get_tensor(out["index"]).argmax() == label)
    return {"size_kb": round(os.path.getsize(path) / 1024, 1), "accuracy": round(correct / n, 4),
            "latency_ms": round(1000 * float(np.mean(times)), 3)}

def bench_keras(model, n=200):
    model(test_images[:1], training=False)
    t0 = time.perf_counter()
    for i in range(n):
        model(test_images[i:i + 1], training=False)
    acc = float((model.predict(test_images[:1000], verbose=0).argmax(1) == test_labels[:1000].flatten()).mean())
    return {"size_kb": round(os.path.getsize("outputs/final_model.keras") / 1024, 1),
            "accuracy": round(acc, 4), "latency_ms": round(1000 * (time.perf_counter() - t0) / n, 3)}

opt = {"keras": bench_keras(model_for_prediction)}
for mode in ["fp32", "fp16", "int8"]:
    opt[f"tflite_{mode}"] = bench_tflite(to_tflite(model_for_prediction, mode))
results["optimisation"] = opt
print(f"{'Variant':<14}{'Size KB':>10}{'Latency ms':>12}{'Accuracy':>10}")
for k, v in opt.items():
    print(f"{k:<14}{v['size_kb']:>10}{v['latency_ms']:>12}{v['accuracy']:>10.2%}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, key, label in zip(ax, ["size_kb", "latency_ms", "accuracy"], ["Size (KB)", "Latency per image (ms)", "Accuracy"]):
    a.bar(list(opt), [opt[k][key] for k in opt], color=["grey", "tab:blue", "tab:orange", "tab:green"])
    a.set_title(label); a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig("outputs/optimisation.png", dpi=130); plt.show()

## 11. Summary for the slides
Prints every number used in the presentation and zips `outputs/` for download. Copy
`cifar10_cnn_int8.tflite` into `webots/controllers/epuck_colour_detector/` to let the robot run the CNN.

In [ ]:
with open("outputs/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
import shutil
shutil.make_archive("cnn_outputs", "zip", "outputs")
try:
    from google.colab import files; files.download("cnn_outputs.zip")
except ImportError:
    print("Saved cnn_outputs.zip")

## 12. Stage 2 – ResNet-9 for maximum accuracy

Runs in the same session after sections 1–9. `stage2_resnet9.py` trains a ResNet-9 (Page, 2018) from scratch: random crop + flip + Cutout, label smoothing 0.1, L2 5e-4, SGD Nesterov with a one-cycle learning rate, mixed precision. It then redoes the confusion matrix, predicts my own images and the e-puck frames both as whole images and as object-centred square crops, and benchmarks ResNet-9 as TFLite fp32/fp16/int8.

Result: **93.1% test accuracy (93.5% with flip TTA)**, 4/4 own photos correct after cropping.

In [ ]:
# upload stage2_resnet9.py to the Colab session first (Files panel)
exec(open('stage2_resnet9.py').read())